In [ ]:
class KNearestNeighbors:
    def __init__(self, k, distance_metric="euclidean"):
        self.k = k
        self.distance_metric = distance_metric

    def fit(self, x, y):
        """
        Just store training set and training labels.
        """
        self.x_train = x       # --> self. becomes public inside the class, every method can access it!!!!!!!!! <--
        self.y_train = y

    def euclidean(self, x_train, x_test):
        """
        Compute the Euclidean distance between two points.
        :param x_train: training set, ndarray, shape = (N, C).
        :param x_test: test set, ndarray, shape = (N, C).
        :return: dist_matrix: ndarray, shape = (N, N).

        step 1: reshape x_train and x_test to have shape (N, 1, C), adding a new dimension = 3D array
        x_train_reshaped
        --> to perform the subtraction using broadcasting
        --> allows to fix x_train row and subtract each row of x_test
        (should be the opposite, but I'm dumb and lazy to fix it, I'll compute this and just traspose it) 
        
        step 2: compute the euclidean distance
        """

        x_train_reshaped = np.expand_dims(x_train, 1)
        x_diff = x_train_reshaped - x_test
        
        dist_matrix = ((x_diff**2).sum(axis=2))**.5
        return dist_matrix
    
    def predict(self, x_test):
        """
        Run the KNN classification on X.
        :param X: input data points, ndarray, shape = (N, C).
        :return: labels: ndarray, shape = (N,).
        """
        
        # STEP 1: compute the distance matrix between text_set (x) and train_set
        distance_matrix = self.euclidean(self.x_train, x_test)
        # in this way the matrix has shape (n_train, n_test) --> to clissify test point I want test points on rows --> turn it upside down, transpose it
        distance_matrix = distance_matrix.T

        # STEP 2: sort per row
        sorted_idx = np.argsort(distance_matrix)  # get the indexes

        # STEP 3: find k nearest neighbours
        k_nearest = sorted_idx[:, :self.k]
        
        # STEP 4: find the corresponding labels to the argsort indexes
        k_nearest_classes = self.y_train[k_nearest]
        
        # STEP 5: assign class based on a majority vote
        predictions = []

        for row in k_nearest_classes:
            values, counts = np.unique(row, return_counts=True)     # ex. values = ["SETOSA", "VERSICOLOR"] ; counts = [2, 1]
            max_idx = np.argmax(counts)
            majority_label = values[max_idx]
            predictions.append(majority_label)

        predictions = np.array(predictions)
        
        return predictions

# read CSV file + create DF
def import_cvs_file (file_path):
    DF = pd.read_csv(file_path, sep=',') #it's a DataFrame
    DF.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']
    return DF

# get test and train data
def get_test_and_train_data(DF):
    DF_shuffled = DF.sample(frac=1, random_state=1) # escamotage to only shuffle the data --> returns all rows, but shuffled.

    # --- 2. Compute split size ---
    n_total = len(DF_shuffled)
    n_test = int(0.2 * n_total)

    # --- 3. Split the dataframe into test and train ---
    test_df = DF_shuffled.iloc[:n_test]
    train_df = DF_shuffled.iloc[n_test:]

    # --- 4. Extract test data x and test labes y ---
    x_test = test_df.drop('class', axis=1).to_numpy()
    y_test = test_df['class'].to_numpy()

    # --- 5. Extract train data x and train labels y ---
    x_train = train_df.drop('class', axis=1).to_numpy()
    y_train = train_df['class'].to_numpy()
    return x_test, y_test, x_train, y_train


if __name__ == '__main__':
    # import csv file
    DF = import_cvs_file('/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB1/iris/iris.data.csv')

    # get train data and test data
    x_test, y_test, x_train, y_train = get_test_and_train_data(DF)

    # create KNearestNeighbors object
    k = 3
    knn = KNearestNeighbors(k)
    knn.fit(x_train, y_train)
    predictions = knn.predict(x_test)

    # visulize properly
    # for idx in range(len(x_test)):
    #     print(f"\nThe prediction for the flower {x_test[idx]} is {predictions[idx]}")
    
    y_pred = knn.predict(x_test)

    # compute accuracy
    count = 0
    for idx in range(len(y_pred)):
        # print(f"\n\nGuessed class: {y_pred[idx]} \nActual class: {y_test[idx]}")
        if y_pred[idx] == y_test[idx]:
            count += 1
        else:
            print(f"\n\nGuessed class: {y_pred[idx]} \nActual class: {y_test[idx]}")
    accuracy = count / len(y_test)
    print(f"Accuracy: {accuracy} %")

#### 11. Load the Ames Housing dataset, select only the numerical features, and define the target variable as House Style. Replace missing values with the mean of each column. Then, split the data into train and test sets.

- Load the Ames Housing dataset
- select only the numerical features --> remove all non-numerical columns from the DS:
> For the selection of the numerical features, you can use pandas.select_dtypes --> DF.select_dtypes(include = np.number)

- define the target variable as House Style ?????
> You’re doing a supervised learning task.

> So you must choose:
- X = the input features (numerical columns only)
- y = the label you want to predict

Here, the instructor is telling you:
> Your label (the thing your model must predict) is the column called HouseStyle in the Ames dataset. Which has values like:
- 1Story
- 2Story
- SLvl
- 1.5Fin
- ...

- Now that you've kept, only the numerical values, replace missing values with the mean of each column
- split the data into train and test sets

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

In [ ]:
def read_csv_file(file_path):
    # read the file
    DF = pd.read_csv(file_path, sep = ',')
    return DF

def only_numeric_DataFrame(DF):
    # keep only numeric columns
    clean_DF = DF.select_dtypes(include = np.number)
    
    # get target variable
    all_labels = DF['House Style']
    
    # manage missing / Nan values 
    all_labels = all_labels.fillna(0)   # 0 = missing
    
    return clean_DF, all_labels

def replace_missing_values_mean(clean_DF):
    # display(clean_DF.info())
    # display(clean_DF.describe())
    # describe = clean_DF.describe()

    for column in clean_DF.columns:
        clean_DF[column] = clean_DF[column].fillna(clean_DF[column].mean())
        # or
        # clean_DF[column] = clean_DF[column].fillna(describe[column]['mean'])
    return clean_DF
    
def train_and_test_set(numerical_DF, all_labels):
    # shuffle the data
    numerical_DF_shuffled = numerical_DF.sample(frac=1, random_state=1)

    # get test data
    n_total = len(numerical_DF_shuffled)
    n_test = int(0.2 * n_total)

    x_test = numerical_DF_shuffled.iloc[:n_test]
    x_test_idx = x_test.index   # even if it's shuffled, the indexes are the original ones, so chill :)
    y_test = all_labels[x_test_idx]
    
    # get train data
    x_train = numerical_DF_shuffled.iloc[n_test:]
    x_train_idx = x_train.index
    y_train = all_labels[x_train_idx]
    
    return x_test, y_test, x_train, y_train



if __name__ == '__main__':
    file_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB1/AmesHousing.csv'
    
    # read file
    DF = read_csv_file(file_path)
    
    # keep only numeric columns
    numeric_DF, all_labels = only_numeric_DataFrame(DF)
    
    # remove replace missing values with mean
    clean_DF = replace_missing_values_mean(numeric_DF)

    # get train and test sets
    x_test, y_test, x_train, y_train = train_and_test_set(clean_DF, all_labels)
    print(f"This is the test set:")
    display(x_test)
    print(f"\n\nThis is the test label:")
    display(y_test)
    print(f"\n\nThis is the train set:")
    display(x_train)
    print(f"\n\nThis is the train label:")
    display(y_train)

---

#### 12. Train your KNN classifier on the dataset and evaluate its accuracy. Then, compare your result with a naive baseline where all houses are predicted as belonging to the majority class (i.e., the most frequent "House Style" in the dataset).

- add these function in the KNearestNeighbors class
- do the fit
- do the prediction
- compare predicted class and the actual class y_test --> compute accyracy

- them compute accuracy as if all classes are the majority class --> naive accuracy
- compare accuracy and naive accuracy

In [ ]:
def read_csv_file(file_path):
    # read the file
    DF = pd.read_csv(file_path, sep = ',')
    return DF

def only_numeric_DataFrame(DF):
    # keep only numeric columns
    clean_DF = DF.select_dtypes(include = np.number)
    
    # get target variable
    all_labels = DF['House Style']
    
    # manage missing / Nan values 
    all_labels = all_labels.fillna(0)   # 0 = missing
    
    return clean_DF, all_labels

def replace_missing_values_mean(clean_DF):
    # display(clean_DF.info())
    # display(clean_DF.describe())
    # describe = clean_DF.describe()

    for column in clean_DF.columns:
        clean_DF[column] = clean_DF[column].fillna(clean_DF[column].mean())
        # or
        # clean_DF[column] = clean_DF[column].fillna(describe[column]['mean'])
    return clean_DF
    
def train_and_test_set(numerical_DF, all_labels):
    # shuffle the data
    numerical_DF_shuffled = numerical_DF.sample(frac=1, random_state=1)

    # get test data
    n_total = len(numerical_DF_shuffled)
    n_test = int(0.2 * n_total)

    x_test = numerical_DF_shuffled.iloc[:n_test]
    x_test_idx = x_test.index   # even if it's shuffled, the indexes are the original ones, so chill :)
    y_test = all_labels[x_test_idx]
    
    # get train data
    x_train = numerical_DF_shuffled.iloc[n_test:]
    x_train_idx = x_train.index
    y_train = all_labels[x_train_idx]
    
    return x_test, y_test, x_train, y_train



if __name__ == '__main__':
    file_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB1/AmesHousing.csv'
    
    # read file
    DF = read_csv_file(file_path)
    
    # keep only numeric columns
    numeric_DF, all_labels = only_numeric_DataFrame(DF)
    
    # remove replace missing values with mean
    clean_DF = replace_missing_values_mean(numeric_DF)

    # get train and test sets
    x_test, y_test, x_train, y_train = train_and_test_set(clean_DF, all_labels)
    print(f"This is the test set:")
    display(x_test)
    print(f"\n\nThis is the test label:")
    display(y_test)
    print(f"\n\nThis is the train set:")
    display(x_train)
    print(f"\n\nThis is the train label:")
    display(y_train)

In [ ]:
class KNearestNeighbors:
    def __init__(self, k, distance_metric="euclidean"):
        self.k = k
        self.distance_metric = distance_metric

    def fit(self, x, y):
        """
        Just store training set and training labels.
        """
        self.x_train = x       # --> self. becomes public inside the class, every method can access it!!!!!!!!! <--
        self.y_train = y

    def euclidean(self, x_train, x_test):
        """
        Compute the Euclidean distance between two points.
        :param x_train: training set, ndarray, shape = (N, C).
        :param x_test: test set, ndarray, shape = (N, C).
        :return: dist_matrix: ndarray, shape = (N, N).

        step 1: reshape x_train and x_test to have shape (N, 1, C), adding a new dimension = 3D array
        x_train_reshaped
        --> to perform the subtraction using broadcasting
        --> allows to fix x_train row and subtract each row of x_test
        (should be the opposite, but I'm dumb and lazy to fix it, I'll compute this and just traspose it) 
        
        step 2: compute the euclidean distance
        """

        x_train_reshaped = np.expand_dims(x_train, 1)
        x_diff = x_train_reshaped - x_test
        
        dist_matrix = ((x_diff**2).sum(axis=2))**.5
        return dist_matrix
    
    def predict(self, x_test):
        """
        Run the KNN classification on X.
        :param X: input data points, ndarray, shape = (N, C).
        :return: labels: ndarray, shape = (N,).
        """
        
        # STEP 1: compute the distance matrix between text_set (x) and train_set
        distance_matrix = self.euclidean(self.x_train, x_test)
        # in this way the matrix has shape (n_train, n_test) --> to clissify test point I want test points on rows --> turn it upside down, transpose it
        distance_matrix = distance_matrix.T

        # STEP 2: sort per row
        sorted_idx = np.argsort(distance_matrix)  # get the indexes

        # STEP 3: find k nearest neighbours
        k_nearest = sorted_idx[:, :self.k]
        
        # STEP 4: find the corresponding labels to the argsort indexes
        k_nearest_classes = self.y_train[k_nearest]
        
        # STEP 5: assign class based on a majority vote
        predictions = []

        for row in k_nearest_classes:
            values, counts = np.unique(row, return_counts=True)     # ex. values = ["SETOSA", "VERSICOLOR"] ; counts = [2, 1]
            max_idx = np.argmax(counts)
            majority_label = values[max_idx]
            predictions.append(majority_label)

        predictions = np.array(predictions)
        
        return predictions
    
    def read_csv_file(self, file_path):
        # read the file
        DF = pd.read_csv(file_path, sep = ',')
        return DF

    def only_numeric_DataFrame(self, DF):
        # keep only numeric columns
        clean_DF = DF.select_dtypes(include = np.number)
        
        # get target variable
        all_labels = DF['House Style']
        
        # manage missing / Nan values 
        all_labels = all_labels.fillna(0)   # 0 = missing
        
        return clean_DF, all_labels

    def replace_missing_values_mean(self, clean_DF):
        # display(clean_DF.info())
        # display(clean_DF.describe())
        # describe = clean_DF.describe()

        for column in clean_DF.columns:
            clean_DF[column] = clean_DF[column].fillna(clean_DF[column].mean())
            # or
            # clean_DF[column] = clean_DF[column].fillna(describe[column]['mean'])
        return clean_DF
        
    def train_and_test_data(self, numerical_DF, all_labels):
        # shuffle the data
        numerical_DF_shuffled = numerical_DF.sample(frac=1, random_state=1)

        # get test data
        n_total = len(numerical_DF_shuffled)
        n_test = int(0.2 * n_total)

        x_test_df = numerical_DF_shuffled.iloc[:n_test]    # is a DF, i want an array --> covert later, I need DF indexes
        x_test_idx = x_test_df.index   # even if it's shuffled, the indexes are the original ones, so chill :)
        y_test_df = all_labels[x_test_idx]
        
        # get train data
        x_train_df = numerical_DF_shuffled.iloc[n_test:]
        x_train_idx = x_train_df.index
        y_train_df = all_labels[x_train_idx]
        

        # convert everything in arrays, do it now and not before, as you need DF indexes
        x_test = x_test_df.to_numpy()
        y_test = y_test_df.to_numpy()
        x_train = x_train_df.to_numpy()
        y_train = y_train_df.to_numpy()
        
        return x_test, y_test, x_train, y_train


############################################################# ENTRY POINT ############################################################################################

if __name__ == '__main__':
    # create object
    knn = KNearestNeighbors(3)
    
    file_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB1/AmesHousing.csv'
    
        # read file
    DF = knn.read_csv_file(file_path)
    
        # keep only numeric columns
    numeric_DF, all_labels = knn.only_numeric_DataFrame(DF)
    
        # remove replace missing values with mean
    clean_DF = knn.replace_missing_values_mean(numeric_DF)

        # get train and test sets
    x_test, y_test, x_train, y_train = knn.train_and_test_data(clean_DF, all_labels)
    
    # visualize test and train data
    # print(f"This is the test set:")
    # display(x_test)
    # print(f"\n\nThis is the test label:")
    # display(y_test)
    # print(f"\n\nThis is the train set:")
    # display(x_train)
    # print(f"\n\nThis is the train label:")
    # display(y_train)

    # fit the model
    knn.fit(x_train, y_train)

    # predict
    predictions = knn.predict(x_test)
    
    # visualize test point, predicted class and actual class. The visualisazion sucks because of Numpy that keep so many decimal numbers and I cannot force it to drop some >:(
    # for idx in range(len(predictions)):
    #     print(f"\n\nThe test point {idx}: \n{x_test[idx]}")
    #     print(f"\n\n Has been predicted as: \n{predictions[idx]}")
    #     print(f"\n\n The actual class is: \n{y_test[idx]}")

    # compute accuracy
    correct_prediction_count = 0
    for idx in range(len(predictions)):
        if predictions[idx] == y_test[idx]:
            correct_prediction_count += 1
        # else:
        #     print("\n WRONG PREDICTION!")
        #     print(f"\n\nThe test point {idx}: \n{x_test[idx]}")
        #     print(f"\n\n Has been predicted as: \n{predictions[idx]}")
        #     print(f"\n\n The actual class is: \n{y_test[idx]}")
    
    accuracy = correct_prediction_count / len(predictions)
    # print(f"Accuracy of the model: {accuracy}")
    # print("It absolutely sucks ass")

    # compute NAIVE accuracy
    max_class = max(y_test)

    # create naive array
    naive_acc_array = []
    for idx in range(len(y_test)):
        naive_acc_array.append(max_class)
    # naive_acc_array = naive_acc_array.to_numpy()    # use to_numpy only to convert DF into arrays, to convert lists use np.array()
    naive_acc_array = np.array(naive_acc_array)

    # compute accuracy

    correct_class_naive_count = 0
    for idx in range(len(y_test)):
        if y_test[idx] == naive_acc_array[idx]:
            correct_class_naive_count +=1
    
    naive_accuracy = correct_class_naive_count / len(y_test)
    # print(f"Accuracy of the naive model: {naive_accuracy} %")

    # compare the accuracy to the benchmark one (naive accuracy)
    if naive_accuracy < accuracy:
        print(f"Thank God at least my model is better than the naive one:")
        print(f"Naive accuracy: {naive_accuracy*100} %")
        print(f"Actual accuracy: {accuracy*100} %")
    else:
        print(f"FUUUUUUCK, the model sucks so much ass that it is worse than the naive one")
        print(f"Naive accuracy: {naive_accuracy*100} %")
        print(f"Actual accuracy: {accuracy*100} %")

    

    

---

#### Ok that the model accuracy is better than the naive one, but ≈60% still sucks ass

- Let's try to improve it by re-training it BUT before re-training it standardize all numbers of the DS so that every number is between 0-1 --> in this way big numbers, I mean literally big mathematical numbers (like 100000), like Lot Config = 1960, has less impact.
- It makes sense as there's absolutely no reason of why Config = 1960 should count more / weight more than a mathematically smaller value.
- Right now, our model evaluates more Lot Config = 1960 than Land Contour = 5 just because it has a bigger value.


> About weights: it would be smart to add weights so that closer neighbours weight more, which makes sense are they're more important.
- Standardizing, on the other hand, removes the meaningless weights caused by the math numbers: a mathematically big number weights more than a mathematically small number and it doesn't make any sense.

#### 13. Before re-training, standardize the dataset to remove scale effects between features. Use the normalization formula:

### $X' = \frac{X - \mu}{\sigma}$

#### where $\mu$ and $\sigma$ are the mean and standard deviation computed on each feature of the training set.
### Train the KNN again and compare the accuracy.
### Is it improved? Why?

- $X' = \frac{X - \mu}{\sigma}$
- $\mu$ = mean value for each attribute (column) of the trainig set
- $\sigma$ = standard deviation for each attribute (column) of the trainig set

So for EACH vaklue in a COLUMN you should:
- take that value
- subtract mean
- divide for standard deviation

- work with DF, DO NOT EVEN THINK OF USING ARRAYS, **FUCK THEM**
- take the column values, which is a series
- compute mean value and standard deviation for that series
- create a new DF stand_DF where each value is standardised: use the formula for each value.

#### Let's normalize

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt

In [ ]:
def read_csv_file(file_path):
    # read the file
    DF = pd.read_csv(file_path, sep = ',')
    return DF

def only_numeric_DataFrame(DF):
    # keep only numeric columns
    clean_DF = DF.select_dtypes(include = np.number)
    
    # get target variable
    all_labels = DF['House Style']
    
    # manage missing / Nan values 
    all_labels = all_labels.fillna(0)   # 0 = missing
    
    return clean_DF, all_labels

def replace_missing_values_mean(clean_DF):
    # display(clean_DF.info())
    # display(clean_DF.describe())
    # describe = clean_DF.describe()

    for column in clean_DF.columns:
        clean_DF[column] = clean_DF[column].fillna(clean_DF[column].mean())
        # or
        # clean_DF[column] = clean_DF[column].fillna(describe[column]['mean'])
    return clean_DF
    
def train_and_test_data(numerical_DF, all_labels):
        # shuffle the data
        numerical_DF_shuffled = numerical_DF.sample(frac=1, random_state=1)

        # get test data
        n_total = len(numerical_DF_shuffled)
        n_test = int(0.2 * n_total)

        x_test_df = numerical_DF_shuffled.iloc[:n_test]    # is a DF, i want an array --> covert later, I need DF indexes
        x_test_idx = x_test_df.index   # even if it's shuffled, the indexes are the original ones, so chill :)
        y_test_df = all_labels[x_test_idx]
        
        # get train data
        x_train_df = numerical_DF_shuffled.iloc[n_test:]
        x_train_idx = x_train_df.index
        y_train_df = all_labels[x_train_idx]
        

        # convert everything in arrays, do it now and not before, as you need DF indexes
        x_test = x_test_df.to_numpy()
        y_test = y_test_df.to_numpy()
        x_train = x_train_df.to_numpy()
        y_train = y_train_df.to_numpy()
        
        return x_test, y_test, x_train, y_train

def before_training_normalization(DF):
    # print("before normalization:")
    # display(DF)

    for column in DF.columns:
        mu = DF[column].mean()
        sigma = DF[column].std()
        DF[column] = (DF[column] - mu) / sigma
    
    # print("After normalization:")
    # display(DF.head(10))
    
    return DF



if __name__ == '__main__':
    file_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB1/AmesHousing.csv'
    
    # read file
    DF = read_csv_file(file_path)
    
    # keep only numeric columns
    numeric_DF, all_labels = only_numeric_DataFrame(DF)
    
    # remove replace missing values with mean
    clean_DF = replace_missing_values_mean(numeric_DF)

    # before getting train and test sets, normalize
    clean_DF_normalized = before_training_normalization(clean_DF)

    # get train and test sets
    x_test, y_test, x_train, y_train = train_and_test_data(clean_DF_normalized, all_labels)


#### Nice, now implement it in the whole code:

In [ ]:
class KNearestNeighbors:
    def __init__(self, k, distance_metric="euclidean"):
        self.k = k
        self.distance_metric = distance_metric

    def fit(self, x, y):
        """
        Just store training set and training labels.
        """
        self.x_train = x       # --> self. becomes public inside the class, every method can access it!!!!!!!!! <--
        self.y_train = y

    def euclidean(self, x_train, x_test):
        """
        Compute the Euclidean distance between two points.
        :param x_train: training set, ndarray, shape = (N, C).
        :param x_test: test set, ndarray, shape = (N, C).
        :return: dist_matrix: ndarray, shape = (N, N).

        step 1: reshape x_train and x_test to have shape (N, 1, C), adding a new dimension = 3D array
        x_train_reshaped
        --> to perform the subtraction using broadcasting
        --> allows to fix x_train row and subtract each row of x_test
        (should be the opposite, but I'm dumb and lazy to fix it, I'll compute this and just traspose it) 
        
        step 2: compute the euclidean distance
        """

        x_train_reshaped = np.expand_dims(x_train, 1)
        x_diff = x_train_reshaped - x_test
        
        dist_matrix = ((x_diff**2).sum(axis=2))**.5
        return dist_matrix
    
    def predict(self, x_test):
        """
        Run the KNN classification on X.
        :param X: input data points, ndarray, shape = (N, C).
        :return: labels: ndarray, shape = (N,).
        """
        
        # STEP 1: compute the distance matrix between text_set (x) and train_set
        distance_matrix = self.euclidean(self.x_train, x_test)
        # in this way the matrix has shape (n_train, n_test) --> to clissify test point I want test points on rows --> turn it upside down, transpose it
        distance_matrix = distance_matrix.T

        # STEP 2: sort per row
        sorted_idx = np.argsort(distance_matrix)  # get the indexes

        # STEP 3: find k nearest neighbours
        k_nearest = sorted_idx[:, :self.k]
        
        # STEP 4: find the corresponding labels to the argsort indexes
        k_nearest_classes = self.y_train[k_nearest]
        
        # STEP 5: assign class based on a majority vote
        predictions = []

        for row in k_nearest_classes:
            values, counts = np.unique(row, return_counts=True)     # ex. values = ["SETOSA", "VERSICOLOR"] ; counts = [2, 1]
            max_idx = np.argmax(counts)
            majority_label = values[max_idx]
            predictions.append(majority_label)

        predictions = np.array(predictions)
        
        return predictions
    
    def read_csv_file(self, file_path):
        # read the file
        DF = pd.read_csv(file_path, sep = ',')
        return DF

    def only_numeric_DataFrame(self, DF):
        # keep only numeric columns
        clean_DF = DF.select_dtypes(include = np.number)
        
        # get target variable
        all_labels = DF['House Style']
        
        # manage missing / Nan values 
        all_labels = all_labels.fillna(0)   # 0 = missing
        
        return clean_DF, all_labels

    def replace_missing_values_mean(self, clean_DF):
        # display(clean_DF.info())
        # display(clean_DF.describe())
        # describe = clean_DF.describe()

        for column in clean_DF.columns:
            clean_DF[column] = clean_DF[column].fillna(clean_DF[column].mean())
            # or
            # clean_DF[column] = clean_DF[column].fillna(describe[column]['mean'])
        return clean_DF
        
    def train_and_test_data(self, numerical_DF, all_labels):
        # shuffle the data
        numerical_DF_shuffled = numerical_DF.sample(frac=1, random_state=1)

        # get test data
        n_total = len(numerical_DF_shuffled)
        n_test = int(0.2 * n_total)

        x_test_df = numerical_DF_shuffled.iloc[:n_test]    # is a DF, i want an array --> covert later, I need DF indexes
        x_test_idx = x_test_df.index   # even if it's shuffled, the indexes are the original ones, so chill :)
        y_test_df = all_labels[x_test_idx]
        
        # get train data
        x_train_df = numerical_DF_shuffled.iloc[n_test:]
        x_train_idx = x_train_df.index
        y_train_df = all_labels[x_train_idx]
        

        # convert everything in arrays, do it now and not before, as you need DF indexes
        x_test = x_test_df.to_numpy()
        y_test = y_test_df.to_numpy()
        x_train = x_train_df.to_numpy()
        y_train = y_train_df.to_numpy()
        
        return x_test, y_test, x_train, y_train

    def before_training_normalization(self, DF):
        # print("before normalization:")
        # display(DF)

        for column in DF.columns:
            mu = DF[column].mean()
            sigma = DF[column].std()
            DF[column] = (DF[column] - mu) / sigma
        
        # print("After normalization:")
        # display(DF.head(10))
        
        return DF

############################################################# ENTRY POINT ############################################################################################

if __name__ == '__main__':
    # create object
    knn = KNearestNeighbors(3)
    
    file_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB1/AmesHousing.csv'
    
        # read file
    DF = knn.read_csv_file(file_path)
    
        # keep only numeric columns
    numeric_DF, all_labels = knn.only_numeric_DataFrame(DF)
    
        # remove replace missing values with mean
    clean_DF = knn.replace_missing_values_mean(numeric_DF)

        # before getting train and test sets, normalize
    clean_DF_normalized = knn.before_training_normalization(clean_DF)

        # get train and test sets
    x_test, y_test, x_train, y_train = knn.train_and_test_data(clean_DF_normalized, all_labels)

    # fit the model
    knn.fit(x_train, y_train)

    # predict
    predictions = knn.predict(x_test)
    
    # visualize test point, predicted class and actual class. The visualisazion sucks because of Numpy that keep so many decimal numbers and I cannot force it to drop some >:(
    # for idx in range(len(predictions)):
    #     print(f"\n\nThe test point {idx}: \n{x_test[idx]}")
    #     print(f"\n\n Has been predicted as: \n{predictions[idx]}")
    #     print(f"\n\n The actual class is: \n{y_test[idx]}")

    # compute accuracy
    correct_prediction_count = 0
    for idx in range(len(predictions)):
        if predictions[idx] == y_test[idx]:
            correct_prediction_count += 1
        # else:
        #     print("\n WRONG PREDICTION!")
        #     print(f"\n\nThe test point {idx}: \n{x_test[idx]}")
        #     print(f"\n\n Has been predicted as: \n{predictions[idx]}")
        #     print(f"\n\n The actual class is: \n{y_test[idx]}")
    
    accuracy = correct_prediction_count / len(predictions)
    # print(f"Accuracy of the model: {accuracy}")
    # print("It absolutely sucks ass")

    # compute NAIVE accuracy
    max_class = max(y_test)

    # create naive array
    naive_acc_array = []
    for idx in range(len(y_test)):
        naive_acc_array.append(max_class)
    # naive_acc_array = naive_acc_array.to_numpy()    # use to_numpy only to convert DF into arrays, to convert lists use np.array()
    naive_acc_array = np.array(naive_acc_array)

    # compute accuracy
    correct_class_naive_count = 0
    for idx in range(len(y_test)):
        if y_test[idx] == naive_acc_array[idx]:
            correct_class_naive_count +=1
    
    naive_accuracy = correct_class_naive_count / len(y_test)
    # print(f"Accuracy of the naive model: {naive_accuracy} %")

    # compare the accuracy to the benchmark one (naive accuracy)
    print(f"New accuracy after normalization: {accuracy*100} %")

---

### 14. Repeat the experiment for different values of k and evaluate how the parameter k affects model performance.

# plot the K values on the X axis and accuracy on the Y axis

In [ ]:
class KNearestNeighbors:
    def __init__(self, k, distance_metric="euclidean"):
        self.k = k
        self.distance_metric = distance_metric

    def fit(self, x, y):
        """
        Just store training set and training labels.
        """
        self.x_train = x       # --> self. becomes public inside the class, every method can access it!!!!!!!!! <--
        self.y_train = y

    def euclidean(self, x_train, x_test):
        """
        Compute the Euclidean distance between two points.
        :param x_train: training set, ndarray, shape = (N, C).
        :param x_test: test set, ndarray, shape = (N, C).
        :return: dist_matrix: ndarray, shape = (N, N).

        step 1: reshape x_train and x_test to have shape (N, 1, C), adding a new dimension = 3D array
        x_train_reshaped
        --> to perform the subtraction using broadcasting
        --> allows to fix x_train row and subtract each row of x_test
        (should be the opposite, but I'm dumb and lazy to fix it, I'll compute this and just traspose it) 
        
        step 2: compute the euclidean distance
        """

        x_train_reshaped = np.expand_dims(x_train, 1)
        x_diff = x_train_reshaped - x_test
        
        dist_matrix = ((x_diff**2).sum(axis=2))**.5
        return dist_matrix
    
    def predict(self, x_test):
        """
        Run the KNN classification on X.
        :param X: input data points, ndarray, shape = (N, C).
        :return: labels: ndarray, shape = (N,).
        """
        
        # STEP 1: compute the distance matrix between text_set (x) and train_set
        distance_matrix = self.euclidean(self.x_train, x_test)
        # in this way the matrix has shape (n_train, n_test) --> to clissify test point I want test points on rows --> turn it upside down, transpose it
        distance_matrix = distance_matrix.T

        # STEP 2: sort per row
        sorted_idx = np.argsort(distance_matrix)  # get the indexes

        # STEP 3: find k nearest neighbours
        k_nearest = sorted_idx[:, :self.k]
        
        # STEP 4: find the corresponding labels to the argsort indexes
        k_nearest_classes = self.y_train[k_nearest]
        
        # STEP 5: assign class based on a majority vote
        predictions = []

        for row in k_nearest_classes:
            values, counts = np.unique(row, return_counts=True)     # ex. values = ["SETOSA", "VERSICOLOR"] ; counts = [2, 1]
            max_idx = np.argmax(counts)
            majority_label = values[max_idx]
            predictions.append(majority_label)

        predictions = np.array(predictions)
        
        return predictions
    
    def read_csv_file(self, file_path):
        # read the file
        DF = pd.read_csv(file_path, sep = ',')
        return DF

    def only_numeric_DataFrame(self, DF):
        # keep only numeric columns
        clean_DF = DF.select_dtypes(include = np.number)
        
        # get target variable
        all_labels = DF['House Style']
        
        # manage missing / Nan values 
        all_labels = all_labels.fillna(0)   # 0 = missing
        
        return clean_DF, all_labels

    def replace_missing_values_mean(self, clean_DF):
        # display(clean_DF.info())
        # display(clean_DF.describe())
        # describe = clean_DF.describe()

        for column in clean_DF.columns:
            clean_DF[column] = clean_DF[column].fillna(clean_DF[column].mean())
            # or
            # clean_DF[column] = clean_DF[column].fillna(describe[column]['mean'])
        return clean_DF
        
    def train_and_test_data(self, numerical_DF, all_labels):
        # shuffle the data
        numerical_DF_shuffled = numerical_DF.sample(frac=1, random_state=1)

        # get test data
        n_total = len(numerical_DF_shuffled)
        n_test = int(0.2 * n_total)

        x_test_df = numerical_DF_shuffled.iloc[:n_test]    # is a DF, i want an array --> covert later, I need DF indexes
        x_test_idx = x_test_df.index   # even if it's shuffled, the indexes are the original ones, so chill :)
        y_test_df = all_labels[x_test_idx]
        
        # get train data
        x_train_df = numerical_DF_shuffled.iloc[n_test:]
        x_train_idx = x_train_df.index
        y_train_df = all_labels[x_train_idx]
        

        # convert everything in arrays, do it now and not before, as you need DF indexes
        x_test = x_test_df.to_numpy()
        y_test = y_test_df.to_numpy()
        x_train = x_train_df.to_numpy()
        y_train = y_train_df.to_numpy()
        
        return x_test, y_test, x_train, y_train

    def before_training_normalization(self, DF):
        # print("before normalization:")
        # display(DF)

        for column in DF.columns:
            mu = DF[column].mean()
            sigma = DF[column].std()
            DF[column] = (DF[column] - mu) / sigma
        
        # print("After normalization:")
        # display(DF.head(10))
        
        return DF

############################################################# ENTRY POINT ############################################################################################

if __name__ == '__main__':
    # create a "dummy" object, just so that I can use the read file, clean the DF and get train and test data, just an access point
    k = 1
    knn = KNearestNeighbors(k)
    
    # file, DF, the cleaned DF, the normalized DF, the test and train data remains ALL the same indipendently of K --> compute it just once
    # then for the results vary k
    
    file_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB1/AmesHousing.csv'
    
        # read file
    DF = knn.read_csv_file(file_path)
    
        # keep only numeric columns
    numeric_DF, all_labels = knn.only_numeric_DataFrame(DF)
    
        # remove replace missing values with mean
    clean_DF = knn.replace_missing_values_mean(numeric_DF)

        # before getting train and test sets, normalize
    clean_DF_normalized = knn.before_training_normalization(clean_DF)

        # get train and test sets
    x_test, y_test, x_train, y_train = knn.train_and_test_data(clean_DF_normalized, all_labels)


    k_values = [1, 5, 8, 9, 10, 15, 20, 25, 50]
    results = {}

    for k in k_values:
        print(f"\n=== Testing k = {k} ===")

        # create knn object that uses different k
        knn = KNearestNeighbors(k)

        # train it
        knn.fit(x_train, y_train)

        # test it
        predictions = knn.predict(x_test)

        # compute accuracy
        correct_predicted_class = 0
        for idx in range(len(predictions)):
            if predictions[idx] == y_test[idx]:
                correct_predicted_class +=1 
        accuracy = correct_predicted_class / len(predictions)

        results[k] = accuracy
        
        print(f"Accuracy with k={k}: {accuracy:.4f}")

    print("\nSummary:", results)

    
    # plot how the accuracy vaires on k
    
    # PLOT 1
    PLOT_1 = False
    if PLOT_1:
        fig, ax = plt.subplots(figsize = (10,5))
        x = results.keys()
        y = results.values()

        ax.bar(x,y)
        # same as:
        # ax.bar(results.keys(), results.values())
        
        plt.show()

    # PLOT 2
    PLOT_2 = True
    if PLOT_2:
        fig, ax = plt.subplots(figsize=(10,5))

        ax.plot(list(results.keys()), list(results.values()), marker='o')
        ax.set_xlabel("k")
        ax.set_ylabel("Accuracy")
        ax.set_title("KNN accuracy vs k")

        plt.grid(True)
        plt.show()

    # PLOT 3
    PLOT_3 = False
    if PLOT_3:
        plt.figure(figsize=(10,5))
        plt.scatter(results.keys(), results.values())
        plt.xlabel("k")
        plt.ylabel("Accuracy")
        plt.title("KNN accuracy vs k")
        plt.grid(True)
        plt.show()


# OR
#### Use: @staticmethod
- Allows you to recall methods inside the KNearestNeighbors class WITHOUT CREATING A KNearestNeighbors OBJECT.
- You can directly access to those class, they're accessible in the whole code

> SO add @staticmethod above all those functions that you would like to recall without creating a KNearestNeighbors object.

> When you recall a static function, so a method inside the KNearestNeighbors class --> write KNearestNeighbors.name_of_the_method

# IMPORTANT
> if you add @staticmethod --> ** REMOVE self FROM THE PARAMETERS OF THE METHOD!!!! **

In [ ]:
class KNearestNeighbors:
    def __init__(self, k, distance_metric="euclidean"):
        self.k = k
        self.distance_metric = distance_metric

    def fit(self, x, y):
        """
        Just store training set and training labels.
        """
        self.x_train = x       # --> self. becomes public inside the class, every method can access it!!!!!!!!! <--
        self.y_train = y

    def euclidean(self, x_train, x_test):
        """
        Compute the Euclidean distance between two points.
        :param x_train: training set, ndarray, shape = (N, C).
        :param x_test: test set, ndarray, shape = (N, C).
        :return: dist_matrix: ndarray, shape = (N, N).

        step 1: reshape x_train and x_test to have shape (N, 1, C), adding a new dimension = 3D array
        x_train_reshaped
        --> to perform the subtraction using broadcasting
        --> allows to fix x_train row and subtract each row of x_test
        (should be the opposite, but I'm dumb and lazy to fix it, I'll compute this and just traspose it) 
        
        step 2: compute the euclidean distance
        """

        x_train_reshaped = np.expand_dims(x_train, 1)
        x_diff = x_train_reshaped - x_test
        
        dist_matrix = ((x_diff**2).sum(axis=2))**.5
        return dist_matrix
    
    def predict(self, x_test):
        """
        Run the KNN classification on X.
        :param X: input data points, ndarray, shape = (N, C).
        :return: labels: ndarray, shape = (N,).
        """
        
        # STEP 1: compute the distance matrix between text_set (x) and train_set
        distance_matrix = self.euclidean(self.x_train, x_test)
        # in this way the matrix has shape (n_train, n_test) --> to clissify test point I want test points on rows --> turn it upside down, transpose it
        distance_matrix = distance_matrix.T

        # STEP 2: sort per row
        sorted_idx = np.argsort(distance_matrix)  # get the indexes

        # STEP 3: find k nearest neighbours
        k_nearest = sorted_idx[:, :self.k]
        
        # STEP 4: find the corresponding labels to the argsort indexes
        k_nearest_classes = self.y_train[k_nearest]
        
        # STEP 5: assign class based on a majority vote
        predictions = []

        for row in k_nearest_classes:
            values, counts = np.unique(row, return_counts=True)     # ex. values = ["SETOSA", "VERSICOLOR"] ; counts = [2, 1]
            max_idx = np.argmax(counts)
            majority_label = values[max_idx]
            predictions.append(majority_label)

        predictions = np.array(predictions)
        
        return predictions
    
    @staticmethod
    def read_csv_file(file_path):
        # read the file
        DF = pd.read_csv(file_path, sep = ',')
        return DF

    @staticmethod
    def only_numeric_DataFrame(DF):
        # keep only numeric columns
        clean_DF = DF.select_dtypes(include = np.number)
        
        # get target variable
        all_labels = DF['House Style']
        
        # manage missing / Nan values 
        all_labels = all_labels.fillna(0)   # 0 = missing
        
        return clean_DF, all_labels

    @staticmethod
    def replace_missing_values_mean(clean_DF):
        # display(clean_DF.info())
        # display(clean_DF.describe())
        # describe = clean_DF.describe()

        for column in clean_DF.columns:
            clean_DF[column] = clean_DF[column].fillna(clean_DF[column].mean())
            # or
            # clean_DF[column] = clean_DF[column].fillna(describe[column]['mean'])
        return clean_DF
        
    @staticmethod
    def train_and_test_data(numerical_DF, all_labels):
        # shuffle the data
        numerical_DF_shuffled = numerical_DF.sample(frac=1, random_state=1)

        # get test data
        n_total = len(numerical_DF_shuffled)
        n_test = int(0.2 * n_total)

        x_test_df = numerical_DF_shuffled.iloc[:n_test]    # is a DF, i want an array --> covert later, I need DF indexes
        x_test_idx = x_test_df.index   # even if it's shuffled, the indexes are the original ones, so chill :)
        y_test_df = all_labels[x_test_idx]
        
        # get train data
        x_train_df = numerical_DF_shuffled.iloc[n_test:]
        x_train_idx = x_train_df.index
        y_train_df = all_labels[x_train_idx]
        

        # convert everything in arrays, do it now and not before, as you need DF indexes
        x_test = x_test_df.to_numpy()
        y_test = y_test_df.to_numpy()
        x_train = x_train_df.to_numpy()
        y_train = y_train_df.to_numpy()
        
        return x_test, y_test, x_train, y_train

    @staticmethod
    def before_training_normalization(DF):
        # print("before normalization:")
        # display(DF)

        for column in DF.columns:
            mu = DF[column].mean()
            sigma = DF[column].std()
            DF[column] = (DF[column] - mu) / sigma
        
        # print("After normalization:")
        # display(DF.head(10))
        
        return DF
    
    @staticmethod
    def plot_results(results):

        # PLOT 1
        PLOT_1 = False
        if PLOT_1:
            fig, ax = plt.subplots(figsize = (10,5))
            x = results.keys()
            y = results.values()

            ax.bar(x,y)
            # same as:
            # ax.bar(results.keys(), results.values())
            
            plt.show()

        # PLOT 2
        PLOT_2 = True
        if PLOT_2:
            fig, ax = plt.subplots(figsize=(10,5))

            ax.plot(list(results.keys()), list(results.values()), marker='o')
            ax.set_xlabel("k")
            ax.set_ylabel("Accuracy")
            ax.set_title("KNN accuracy vs k")

            plt.grid(True)
            plt.show()

        # PLOT 3
        PLOT_3 = False
        if PLOT_3:
            plt.figure(figsize=(10,5))
            plt.scatter(results.keys(), results.values())
            plt.xlabel("k")
            plt.ylabel("Accuracy")
            plt.title("KNN accuracy vs k")
            plt.grid(True)
            plt.show()

        
######################################################################################################################################################################
############################################################# ENTRY POINT ############################################################################################
######################################################################################################################################################################

if __name__ == '__main__':
        
    # ONE-TIME preprocessing (no dummy object needed) --> use static method functions
    file_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB1/AmesHousing.csv'
    
        # read file
    DF = KNearestNeighbors.read_csv_file(file_path)
    
        # keep only numeric columns
    numeric_DF, all_labels = KNearestNeighbors.only_numeric_DataFrame(DF)
    
        # remove replace missing values with mean
    clean_DF = KNearestNeighbors.replace_missing_values_mean(numeric_DF)

        # before getting train and test sets, normalize
    clean_DF_normalized = KNearestNeighbors.before_training_normalization(clean_DF)

        # get train and test sets
    x_test, y_test, x_train, y_train = KNearestNeighbors.train_and_test_data(clean_DF_normalized, all_labels)

    # NOW we create the KNearestNeighbors object to recall fit and predict

    k_values = [1, 5, 8, 9, 10, 15, 20, 25, 50]
    results = {}

    for k in k_values:
        print(f"\n=== Testing k = {k} ===")

        # create knn object that uses different k
        knn = KNearestNeighbors(k)

        # train it
        knn.fit(x_train, y_train)

        # test it
        predictions = knn.predict(x_test)

        # compute accuracy
        correct_predicted_class = 0
        for idx in range(len(predictions)):
            if predictions[idx] == y_test[idx]:
                correct_predicted_class +=1 
        accuracy = correct_predicted_class / len(predictions)

        results[k] = accuracy
        
        print(f"Accuracy with k={k}: {accuracy:.4f}")

    print("\nSummary:", results)

    
    # plot how the accuracy vaires on k
    KNearestNeighbors.plot_results(results)


---

#### 15. (*) For each value of k, repeat the random split between training and testing data multiple times (e.g., 20 runs). Compute the mean and standard deviation of the accuracy across runs to evaluate the model’s stability.

- Right now I worte in the static method train_and_test_data:
numerical_DF_shuffled = numerical_DF.sample(frac=1, random_state=1)
- Since I want a random split --> remove random_state=1

"FOR EACH VALUE OF K":
- so outer loop where you fix k and then iterate it
- inner loop where fixed k=1 you split into train and test set like 20 times
- each iteration save the accuracy value in like a dictionary with iteration_number : accuracy
- then do the mean and std of these values

- THIS MUST BE REPEATE FOR EACH VALUE OF K!

In [ ]:
class KNearestNeighbors:
    def __init__(self, k, distance_metric="euclidean"):
        self.k = k
        self.distance_metric = distance_metric

    def fit(self, x, y):
        """
        Just store training set and training labels.
        """
        self.x_train = x       # --> self. becomes public inside the class, every method can access it!!!!!!!!! <--
        self.y_train = y

    def euclidean(self, x_train, x_test):
        """
        Compute the Euclidean distance between two points.
        :param x_train: training set, ndarray, shape = (N, C).
        :param x_test: test set, ndarray, shape = (N, C).
        :return: dist_matrix: ndarray, shape = (N, N).

        step 1: reshape x_train and x_test to have shape (N, 1, C), adding a new dimension = 3D array
        x_train_reshaped
        --> to perform the subtraction using broadcasting
        --> allows to fix x_train row and subtract each row of x_test
        (should be the opposite, but I'm dumb and lazy to fix it, I'll compute this and just traspose it) 
        
        step 2: compute the euclidean distance
        """

        x_train_reshaped = np.expand_dims(x_train, 1)
        x_diff = x_train_reshaped - x_test
        
        dist_matrix = ((x_diff**2).sum(axis=2))**.5
        return dist_matrix
    
    def predict(self, x_test):
        """
        Run the KNN classification on X.
        :param X: input data points, ndarray, shape = (N, C).
        :return: labels: ndarray, shape = (N,).
        """
        
        # STEP 1: compute the distance matrix between text_set (x) and train_set
        distance_matrix = self.euclidean(self.x_train, x_test)
        # in this way the matrix has shape (n_train, n_test) --> to clissify test point I want test points on rows --> turn it upside down, transpose it
        distance_matrix = distance_matrix.T

        # STEP 2: sort per row
        sorted_idx = np.argsort(distance_matrix)  # get the indexes

        # STEP 3: find k nearest neighbours
        k_nearest = sorted_idx[:, :self.k]
        
        # STEP 4: find the corresponding labels to the argsort indexes
        k_nearest_classes = self.y_train[k_nearest]
        
        # STEP 5: assign class based on a majority vote
        predictions = []

        for row in k_nearest_classes:
            values, counts = np.unique(row, return_counts=True)     # ex. values = ["SETOSA", "VERSICOLOR"] ; counts = [2, 1]
            max_idx = np.argmax(counts)
            majority_label = values[max_idx]
            predictions.append(majority_label)

        predictions = np.array(predictions)
        
        return predictions
    
    @staticmethod
    def read_csv_file(file_path):
        # read the file
        DF = pd.read_csv(file_path, sep = ',')
        return DF

    @staticmethod
    def only_numeric_DataFrame(DF):
        # keep only numeric columns
        clean_DF = DF.select_dtypes(include = np.number)
        
        # get target variable
        all_labels = DF['House Style']
        
        # manage missing / Nan values 
        all_labels = all_labels.fillna(0)   # 0 = missing
        
        return clean_DF, all_labels

    @staticmethod
    def replace_missing_values_mean(clean_DF):
        # display(clean_DF.info())
        # display(clean_DF.describe())
        # describe = clean_DF.describe()

        for column in clean_DF.columns:
            clean_DF[column] = clean_DF[column].fillna(clean_DF[column].mean())
            # or
            # clean_DF[column] = clean_DF[column].fillna(describe[column]['mean'])
        return clean_DF
        
    @staticmethod
    def train_and_test_data(numerical_DF, all_labels):
        # shuffle the data
        # REMOVE random_state = 1    !!!!!!
        numerical_DF_shuffled = numerical_DF.sample(frac=1)

        # get test data
        n_total = len(numerical_DF_shuffled)
        n_test = int(0.2 * n_total)

        x_test_df = numerical_DF_shuffled.iloc[:n_test]    # is a DF, i want an array --> covert later, I need DF indexes
        x_test_idx = x_test_df.index   # even if it's shuffled, the indexes are the original ones, so chill :)
        y_test_df = all_labels[x_test_idx]
        
        # get train data
        x_train_df = numerical_DF_shuffled.iloc[n_test:]
        x_train_idx = x_train_df.index
        y_train_df = all_labels[x_train_idx]
        

        # convert everything in arrays, do it now and not before, as you need DF indexes
        x_test = x_test_df.to_numpy()
        y_test = y_test_df.to_numpy()
        x_train = x_train_df.to_numpy()
        y_train = y_train_df.to_numpy()
        
        return x_test, y_test, x_train, y_train

    @staticmethod
    def before_training_normalization(DF):
        # print("before normalization:")
        # display(DF)

        for column in DF.columns:
            mu = DF[column].mean()
            sigma = DF[column].std()
            DF[column] = (DF[column] - mu) / sigma
        
        # print("After normalization:")
        # display(DF.head(10))
        
        return DF
    
    @staticmethod
    def plot_results(results):

        # PLOT 1
        PLOT_1 = False
        if PLOT_1:
            fig, ax = plt.subplots(figsize = (10,5))
            x = results.keys()
            y = results.values()

            ax.bar(x,y)
            # same as:
            # ax.bar(results.keys(), results.values())
            
            plt.show()

        # PLOT 2
        PLOT_2 = True
        if PLOT_2:
            fig, ax = plt.subplots(figsize=(10,5))

            ax.plot(list(results.keys()), list(results.values()), marker='o')
            ax.set_xlabel("k")
            ax.set_ylabel("Accuracy")
            ax.set_title("KNN accuracy vs k")

            plt.grid(True)
            plt.show()

        # PLOT 3
        PLOT_3 = False
        if PLOT_3:
            plt.figure(figsize=(10,5))
            plt.scatter(results.keys(), results.values())
            plt.xlabel("k")
            plt.ylabel("Accuracy")
            plt.title("KNN accuracy vs k")
            plt.grid(True)
            plt.show()

        
######################################################################################################################################################################
############################################################# ENTRY POINT ############################################################################################
######################################################################################################################################################################

if __name__ == '__main__':
        
        # ONE-TIME preprocessing (no dummy object needed) --> use static method functions
    file_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB1/AmesHousing.csv'
    
        # read file
    DF = KNearestNeighbors.read_csv_file(file_path)
    
        # keep only numeric columns
    numeric_DF, all_labels = KNearestNeighbors.only_numeric_DataFrame(DF)
    
        # remove replace missing values with mean
    clean_DF = KNearestNeighbors.replace_missing_values_mean(numeric_DF)

        # before getting train and test sets, normalize
    clean_DF_normalized = KNearestNeighbors.before_training_normalization(clean_DF)



        # for each k value
    k_values = [1, 5, 8, 9, 10, 15, 20, 25, 50]
    
    for k in k_values:
        print(f"\n=== Testing k = {k} ===")
        # create knn object with a fixed k
        knn = KNearestNeighbors(k)

        # compute train and test data 20 times FOR EACH VALUE OF K

        iterations = 21
        accuracies_per_iteration = []

        for iteration in range(iterations):
            
            x_test, y_test, x_train, y_train = KNearestNeighbors.train_and_test_data(clean_DF_normalized, all_labels)

                # train it
            knn.fit(x_train, y_train)

                # test it
            predictions = knn.predict(x_test)

                # compute accuracy
            correct_predicted_class = 0
            for idx in range(len(predictions)):
                if predictions[idx] == y_test[idx]:
                    correct_predicted_class +=1 
            accuracy = correct_predicted_class / len(predictions)
            
            accuracies_per_iteration.append(accuracy)
        
        mean_accuracy_per_k = np.mean(accuracies_per_iteration)
        std__accuracy_per_k = np.std(accuracies_per_iteration)

        print(f"\nFor value of k = {k} \nWith {iterations} iterations \nMean accuracy: {mean_accuracy_per_k*100} % \nStandard deviation of the accuracy: {std__accuracy_per_k}\n")

---

#### 16. Rerun the KNN experiments using the different distance definitions implemented (euclidean, cosine, and manhattan distances). For each distance metric, vary kand record the achieved accuracy. Identify which combination of distance metric and kachieves the best overall performance.

# AAAWWW HEEL NAAAAHHHH, LONG AND BORING AF 